# Price Overlay Data Process

This notebook replaces only `data/processed/prosumer/price.csv` values. It keeps the existing 2019-2020 timestamps so the rest of the project can continue using the same load/PV time axis and config years.

Mapping:
- 2021 prices replace 2019 prices, dropping the first four 2021 points because current 2019 starts at 01:00.
- 2022 prices replace 2020 prices, inserting a duplicated 2022-02-28 block for 2020-02-29.
- Source row order is preserved; DST duplicate/missing local timestamps are not matched by key.


In [1]:
from __future__ import annotations

from datetime import datetime
from pathlib import Path
import shutil
import subprocess
import sys

import numpy as np
import pandas as pd

LOCAL_TZ = "Europe/Berlin"
RAW_RELATIVE_PATH = Path("data/raw/Germany_price_15_2021-2022.csv")
PRICE_RELATIVE_PATH = Path("data/processed/prosumer/price.csv")
EXPECTED_SOURCE_YEAR_ROWS = {2021: 35040, 2022: 35040}
EXPECTED_TARGET_YEAR_ROWS = {2019: 35036, 2020: 35136}
EXPECTED_TOTAL_ROWS = 70172


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "configs/experiment_config.py").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Could not locate the MADRL_ESS project root from the current working directory.")


ROOT = find_project_root(Path.cwd())
RAW_PATH = ROOT / RAW_RELATIVE_PATH
PRICE_PATH = ROOT / PRICE_RELATIVE_PATH

if not RAW_PATH.exists():
    raise FileNotFoundError(f"Missing source price file: {RAW_PATH}")
if not PRICE_PATH.exists():
    raise FileNotFoundError(f"Missing processed price file: {PRICE_PATH}")

print(f"Project root: {ROOT}")
print(f"Source raw price file: {RAW_PATH}")
print(f"Processed price file: {PRICE_PATH}")


Project root: C:\Users\10856\Desktop\GithubProject\MADRL_ESS
Source raw price file: C:\Users\10856\Desktop\GithubProject\MADRL_ESS\data\raw\Germany_price_15_2021-2022.csv
Processed price file: C:\Users\10856\Desktop\GithubProject\MADRL_ESS\data\processed\prosumer\price.csv


In [2]:
def load_source_prices(path: Path) -> pd.DataFrame:
    raw = pd.read_csv(path, sep=";", low_memory=False)
    price_columns = [column for column in raw.columns if str(column).startswith("Germany/Luxembourg")]
    if len(price_columns) != 1:
        raise ValueError(f"Expected exactly one Germany/Luxembourg price column, got {price_columns}")
    price_col = price_columns[0]

    source = pd.DataFrame(
        {
            "source_start": pd.to_datetime(raw["Start date"], format="%b %d, %Y %I:%M %p", errors="raise"),
            "price": pd.to_numeric(raw[price_col].replace("-", pd.NA), errors="coerce") / 1000.0,
        }
    )
    if source["price"].isna().any():
        bad_rows = source[source["price"].isna()].head(10)
        raise ValueError(f"Source price contains NaN after parsing. First bad rows:\n{bad_rows}")
    source["source_year"] = source["source_start"].dt.year.astype(int)
    return source


def load_target_prices(path: Path) -> tuple[pd.DataFrame, pd.Series]:
    target = pd.read_csv(path)
    if list(target.columns) != ["timestamp", "price"]:
        raise ValueError(f"Processed price CSV must have columns ['timestamp', 'price'], got {list(target.columns)}")
    target_ts = pd.to_datetime(target["timestamp"], utc=True).dt.tz_convert(LOCAL_TZ)
    return target, target_ts


source = load_source_prices(RAW_PATH)
target, target_ts = load_target_prices(PRICE_PATH)

source_year_counts = source["source_year"].value_counts().sort_index().to_dict()
target_year_counts = target_ts.dt.year.value_counts().sort_index().to_dict()

if source_year_counts != EXPECTED_SOURCE_YEAR_ROWS:
    raise ValueError(f"Unexpected source rows by year: expected={EXPECTED_SOURCE_YEAR_ROWS}, actual={source_year_counts}")
if target_year_counts != EXPECTED_TARGET_YEAR_ROWS:
    raise ValueError(f"Unexpected target rows by year: expected={EXPECTED_TARGET_YEAR_ROWS}, actual={target_year_counts}")
if len(target) != EXPECTED_TOTAL_ROWS:
    raise ValueError(f"Unexpected target row count: expected={EXPECTED_TOTAL_ROWS}, actual={len(target)}")

print("Source rows by year:", source_year_counts)
print("Target rows by year:", target_year_counts)
print("Source price min/max EUR/kWh:", float(source["price"].min()), float(source["price"].max()))


Source rows by year: {2021: 35040, 2022: 35040}
Target rows by year: {2019: 35036, 2020: 35136}
Source price min/max EUR/kWh: -0.069 0.871


In [3]:
def source_year_frame(year: int) -> pd.DataFrame:
    frame = source[source["source_year"].eq(int(year))].reset_index(drop=True)
    if len(frame) != EXPECTED_SOURCE_YEAR_ROWS[int(year)]:
        raise ValueError(f"Unexpected row count for source year {year}: {len(frame)}")
    return frame


source_2021 = source_year_frame(2021)
source_2022 = source_year_frame(2022)

target_2019_mask = target_ts.dt.year.eq(2019).to_numpy()
target_2020_mask = target_ts.dt.year.eq(2020).to_numpy()
target_2019_count = int(target_2019_mask.sum())
target_2020_count = int(target_2020_mask.sum())

# Current 2019 starts at 01:00, so drop the first four 15-minute source points.
replacement_2019 = source_2021.loc[4 : 4 + target_2019_count - 1, "price"].reset_index(drop=True)

# 2020 is a leap year. Insert a duplicated 2022-02-28 block for 2020-02-29.
feb28_2022_mask = source_2022["source_start"].dt.date == pd.Timestamp("2022-02-28").date()
feb28_2022 = source_2022.loc[feb28_2022_mask, "price"].reset_index(drop=True)
if len(feb28_2022) != 96:
    raise ValueError(f"Expected 96 rows for 2022-02-28, got {len(feb28_2022)}")

before_2022_mar01_mask = source_2022["source_start"] < pd.Timestamp("2022-03-01")
before_mar01 = source_2022.loc[before_2022_mar01_mask, "price"].reset_index(drop=True)
from_mar01 = source_2022.loc[~before_2022_mar01_mask, "price"].reset_index(drop=True)
replacement_2020 = pd.concat([before_mar01, feb28_2022, from_mar01], ignore_index=True)

if len(replacement_2019) != target_2019_count:
    raise ValueError(f"2019 replacement length mismatch: {len(replacement_2019)} vs {target_2019_count}")
if len(replacement_2020) != target_2020_count:
    raise ValueError(f"2020 replacement length mismatch: {len(replacement_2020)} vs {target_2020_count}")

print("Replacement lengths:", {"2019": len(replacement_2019), "2020": len(replacement_2020)})
print("2019 first replacement source timestamp:", source_2021.loc[4, "source_start"])
print("2020 first replacement source timestamp:", source_2022.loc[0, "source_start"])
print("2020 leap-day replacement source date: 2022-02-28")


Replacement lengths: {'2019': 35036, '2020': 35136}
2019 first replacement source timestamp: 2021-01-01 01:00:00
2020 first replacement source timestamp: 2022-01-01 00:00:00
2020 leap-day replacement source date: 2022-02-28


In [4]:
backup_path = PRICE_PATH.with_name(f"price.backup_before_2021_2022_overlay_{datetime.now():%Y%m%d_%H%M%S}.csv")
shutil.copy2(PRICE_PATH, backup_path)

output = target.copy()
output.loc[target_2019_mask, "price"] = replacement_2019.to_numpy(dtype=float)
output.loc[target_2020_mask, "price"] = replacement_2020.to_numpy(dtype=float)

temp_path = PRICE_PATH.with_name(f".{PRICE_PATH.stem}.overlay_tmp_{datetime.now():%Y%m%d_%H%M%S}{PRICE_PATH.suffix}")
output.to_csv(temp_path, index=False, float_format="%.8g")
try:
    temp_path.replace(PRICE_PATH)
except PermissionError as exc:
    def ps_quote(path: Path) -> str:
        return "'" + str(path).replace("'", "''") + "'"

    ps_script = f"Move-Item -LiteralPath {ps_quote(temp_path)} -Destination {ps_quote(PRICE_PATH)} -Force"
    result = subprocess.run(
        ["powershell", "-NoProfile", "-ExecutionPolicy", "Bypass", "-Command", ps_script],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        temp_path.unlink(missing_ok=True)
        raise PermissionError(
            f"Could not replace {PRICE_PATH}. Close any editor, spreadsheet, or file preview that has it open, "
            f"then rerun this notebook. PowerShell fallback stderr: {result.stderr}"
        ) from exc

print(f"Backup written: {backup_path}")
print(f"Overlay written: {PRICE_PATH}")


Backup written: C:\Users\10856\Desktop\GithubProject\MADRL_ESS\data\processed\prosumer\price.backup_before_2021_2022_overlay_20260423_150636.csv
Overlay written: C:\Users\10856\Desktop\GithubProject\MADRL_ESS\data\processed\prosumer\price.csv


In [5]:
backup = pd.read_csv(backup_path)
written, written_ts = load_target_prices(PRICE_PATH)

if not written["timestamp"].equals(backup["timestamp"]):
    raise ValueError("Timestamp column changed during overlay; this notebook must preserve target timestamps exactly.")
if list(written.columns) != ["timestamp", "price"]:
    raise ValueError(f"Unexpected written columns: {list(written.columns)}")
if len(written) != EXPECTED_TOTAL_ROWS:
    raise ValueError(f"Unexpected written row count: {len(written)}")
if written["price"].isna().any():
    raise ValueError("Written price column contains NaN values.")

first_2019_idx = int(np.flatnonzero(written_ts.dt.year.eq(2019).to_numpy())[0])
first_2020_idx = int(np.flatnonzero(written_ts.dt.year.eq(2020).to_numpy())[0])
if not np.isclose(float(written.loc[first_2019_idx, "price"]), float(source_2021.loc[4, "price"])):
    raise ValueError("2019 first-price spot check failed.")
if not np.isclose(float(written.loc[first_2020_idx, "price"]), float(source_2022.loc[0, "price"])):
    raise ValueError("2020 first-price spot check failed.")

leap_day_mask = written_ts.dt.date.eq(pd.Timestamp("2020-02-29").date()).to_numpy()
leap_day_values = written.loc[leap_day_mask, "price"].reset_index(drop=True)
if len(leap_day_values) != 96:
    raise ValueError(f"Expected 96 written rows for 2020-02-29, got {len(leap_day_values)}")
if not np.allclose(leap_day_values.to_numpy(dtype=float), feb28_2022.to_numpy(dtype=float), rtol=1e-9, atol=1e-12):
    raise ValueError("2020-02-29 spot check failed; values do not match copied 2022-02-28 block.")

summary = {
    "rows": int(len(written)),
    "price_min_eur_per_kwh": float(written["price"].min()),
    "price_max_eur_per_kwh": float(written["price"].max()),
    "backup_path": str(backup_path),
    "written_path": str(PRICE_PATH),
}
summary


{'rows': 70172,
 'price_min_eur_per_kwh': -0.069,
 'price_max_eur_per_kwh': 0.871,
 'backup_path': 'C:\\Users\\10856\\Desktop\\GithubProject\\MADRL_ESS\\data\\processed\\prosumer\\price.backup_before_2021_2022_overlay_20260423_150636.csv',
 'written_path': 'C:\\Users\\10856\\Desktop\\GithubProject\\MADRL_ESS\\data\\processed\\prosumer\\price.csv'}

In [6]:
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from configs.experiment_config import ExperimentConfig
from data.loaders.registry import build_dataset

cfg = ExperimentConfig()
cfg.data.data_dir = ROOT / "data"

train_dataset = build_dataset(cfg, mode="train")
test_dataset = build_dataset(cfg, mode="test")
train_episode = train_dataset.get_episode(0)
test_episode = test_dataset.get_episode(0)

smoke_summary = {
    "train_episodes": int(train_dataset.num_episodes()),
    "test_episodes": int(test_dataset.num_episodes()),
    "train_first_timestamp": train_episode["meta"]["timestamps"][0],
    "test_first_timestamp": test_episode["meta"]["timestamps"][0],
    "train_first_prices": train_episode["signals"]["wholesale_price"][:4].tolist(),
    "test_first_prices": test_episode["signals"]["wholesale_price"][:4].tolist(),
}
smoke_summary


{'train_episodes': 364,
 'test_episodes': 30,
 'train_first_timestamp': '2019-01-01 01:00:00+01:00',
 'test_first_timestamp': '2020-06-01 00:00:00+02:00',
 'train_first_prices': [0.048190001398324966,
  0.048190001398324966,
  0.048190001398324966,
  0.048190001398324966],
 'test_first_prices': [0.2199999988079071,
  0.2199999988079071,
  0.2199999988079071,
  0.2199999988079071]}